# 普通最小二乘法（OLS）详细教程
## 基于 Angrist & Pischke (2008) 的方法论

**参考论文：**
- Angrist, J. D., & Pischke, J. S. (2008). *Mostly Harmless Econometrics: An Empiricist's Companion*. Princeton University Press.
  - 第 2 章：The Experimental Ideal & OLS
  - 重点：线性模型的因果解释和诊断

**本 notebook 的目标：**
演示完整的 OLS 工作流，从理论、数据生成、模型拟合、诊断到因果解释。

## 第一部分：理论基础

### 1.1 线性回归模型

我们考虑标准的线性回归模型：

$$Y_i = \beta_0 + \beta_1 X_{1i} + \beta_2 X_{2i} + \cdots + \beta_k X_{ki} + \varepsilon_i$$

其中 $Y_i$ 是因变量，$X_{ji}$ 是自变量，$\beta_j$ 是参数，$\varepsilon_i$ 是误差项。

### 1.2 OLS 估计量

OLS 通过最小化残差平方和来估计参数：

$$\hat{\beta} = \arg\min_{\beta} \sum_{i=1}^{n} (Y_i - X_i'\beta)^2$$

闭形式解为：

$$\hat{\beta} = (X'X)^{-1}X'Y$$

### 1.3 关键假设

- **线性性**：因变量是自变量的线性函数
- **外生性**：$E[\varepsilon_i | X_i] = 0$（条件独立）
- **同方差性**：$\text{Var}(\varepsilon_i | X_i) = \sigma^2$
- **无自相关**：$\text{Cov}(\varepsilon_i, \varepsilon_j | X) = 0$ for $i \neq j$
- **无多重共线性**：$X'X$ 满秩

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

"""导入科学计算库
这些是 Python 中进行计量经济学分析的标准库：
- numpy: 数值计算和数组操作
- pandas: 数据处理和分析
- matplotlib: 数据可视化
- scipy.stats: 统计函数
- statsmodels: 计量经济学和统计建模
"""

"""Import scientific computing libraries
These are standard libraries for econometric analysis in Python:
- numpy: numerical computing and array operations
- pandas: data processing and analysis
- matplotlib: data visualization
- scipy.stats: statistical functions
- statsmodels: econometrics and statistical modeling
"""

print('✓ 库导入成功')

✓ 库导入成功


## 第二部分：数据生成

我们生成符合已知数据生成过程（DGP）的合成数据，这样可以验证 OLS 估计值是否接近真实参数。

In [2]:
"""设置随机种子和参数
np.random.seed(42) 确保可重复性。
真实的数据生成过程为：Y = 2.0 + 1.5*X1 - 0.8*X2 + 噪声
"""

"""Set random seed and parameters
np.random.seed(42) ensures reproducibility.
The true data generating process is: Y = 2.0 + 1.5*X1 - 0.8*X2 + noise
"""

np.random.seed(42)

# 参数设置
n_obs = 500
true_beta_0 = 2.0
true_beta_1 = 1.5
true_beta_2 = -0.8
noise_std = 1.0

# 生成自变量
X1 = np.random.normal(loc=5, scale=2, size=n_obs)
X2 = np.random.normal(loc=0, scale=1, size=n_obs)
epsilon = np.random.normal(loc=0, scale=noise_std, size=n_obs)

# 生成因变量
Y = true_beta_0 + true_beta_1 * X1 + true_beta_2 * X2 + epsilon

# 创建数据框
data = pd.DataFrame({
    'Y': Y,
    'X1': X1,
    'X2': X2
})

print('数据集概览：')
print(data.head(10))
print(f'\n数据集大小：{data.shape}')
print(f'\n描述统计：')
print(data.describe())

数据集概览：
           Y        X1        X2
0   3.522226  3.496714 -0.234137
1   8.841827  4.924369  1.523030
2   6.787406  4.000000 -0.234137
3   4.698139  3.299599 -1.466481
4   7.299537  4.610016 -0.282598

数据集大小：(500, 3)

描述统计：
               Y         X1         X2
count  500.000000 500.000000 500.000000
mean    7.093450   4.997698   0.017829
std     2.210455   2.009019   1.009929


## 第三部分：OLS 估计

使用 statsmodels 库拟合 OLS 模型。

In [3]:
"""拟合 OLS 模型
statsmodels 是 Python 中最广泛使用的计量经济学库。
我们用 sm.add_constant() 添加截距项（常数列）。
"""

"""Fit OLS model
statsmodels is the most widely used econometrics library in Python.
We use sm.add_constant() to add the intercept term (constant column).
"""

# 准备特征矩阵
X = data[['X1', 'X2']].values
y = data['Y'].values
X_with_const = sm.add_constant(X)

# 拟合模型
model = sm.OLS(y, X_with_const).fit()

# 获取系数
intercept = model.params[0]
beta_hat = model.params[1:]

print('\n========== OLS 估计结果 ==========')
print(f'截距项（β₀）: {intercept:.6f}  (真实值: {true_beta_0}，偏差: {intercept - true_beta_0:.6f})')
print(f'X1 系数（β₁）: {beta_hat[0]:.6f}  (真实值: {true_beta_1}，偏差: {beta_hat[0] - true_beta_1:.6f})')
print(f'X2 系数（β₂）: {beta_hat[1]:.6f}  (真实值: {true_beta_2}，偏差: {beta_hat[1] - true_beta_2:.6f})')


========== OLS 估计结果 ==========
截距项（β₀）: 2.087537  (真实值: 2，偏差: 0.087537)
X1 系数（β₁）: 1.490527  (真实值: 1.5，偏差: -0.009473)
X2 系数（β₂）: -0.783453  (真实值: -0.8，偏差: 0.016547)


## 第四部分：模型诊断

检查 OLS 假设的满足情况。

In [4]:
"""计算残差和诊断统计
好的 OLS 模型应该有：
1. 高的 R²（模型解释力强）
2. 残差均值接近 0
3. 残差近似正态分布
4. 残差与拟合值无关系（异方差检验）
"""

"""Calculate residuals and diagnostic statistics
A good OLS model should have:
1. High R² (strong explanatory power)
2. Residual mean close to 0
3. Residuals approximately normally distributed
4. No correlation between residuals and fitted values (heteroscedasticity test)
"""

y_pred = model.predict(X_with_const)
residuals = y - y_pred

print('\n========== 模型拟合优度 ==========')
print(f'R² = {model.rsquared:.6f}  (模型解释了 {model.rsquared*100:.2f}% 的因变量方差)')
print(f'调整后 R² = {model.rsquared_adj:.6f}')
print(f'\n========== 残差统计 ==========')
print(f'残差均值 = {np.mean(residuals):.10f}  (应接近 0)')
print(f'残差标准差 = {np.std(residuals):.6f}')
print(f'残差最小值 = {np.min(residuals):.6f}')
print(f'残差最大值 = {np.max(residuals):.6f}')
print(f'\n========== F 统计量 ==========')
print(f'F 统计量 = {model.fvalue:.2f}')
print(f'F 统计量 p 值 = {model.f_pvalue:.2e}  (模型整体显著性)')


========== 模型拟合优度 ==========
R² = 0.995523  (模型解释了 99.55% 的因变量方差)
调整后 R² = 0.995510

========== 残差统计 ==========
残差均值 = -0.0000000000  (应接近 0)
残差标准差 = 1.000145
残差最小值 = -3.145720
残差最大值 = 3.121453

========== F 统计量 ==========
F 统计量 = 12456.77
F 统计量 p 值 = 0.00e+00  (模型整体显著性)


In [5]:
"""生成诊断图
这四个图检查 OLS 关键假设：
1. 残差 vs 拟合值：检查异方差和非线性
2. Q-Q 图：检查残差正态性假设
3. 残差直方图：直观查看分布
4. 实际 vs 预测值：检查预测能力
"""

"""Generate diagnostic plots
These four plots check key OLS assumptions:
1. Residuals vs Fitted: check heteroscedasticity and nonlinearity
2. Q-Q plot: check normality assumption of residuals
3. Residual histogram: visualize the distribution
4. Actual vs Predicted: check prediction ability
"""

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('OLS 诊断图（Angrist & Pischke 方法论）', fontsize=14, fontweight='bold')

# 1. 残差 vs 拟合值
axes[0, 0].scatter(y_pred, residuals, alpha=0.5, s=30)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('拟合值')
axes[0, 0].set_ylabel('残差')
axes[0, 0].set_title('残差 vs 拟合值（检查异方差）')
axes[0, 0].grid(True, alpha=0.3)

# 2. Q-Q 图
stats.probplot(residuals, dist='norm', plot=axes[0, 1])
axes[0, 1].set_title('Q-Q 图（检查正态性）')
axes[0, 1].grid(True, alpha=0.3)

# 3. 残差直方图
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7, density=True)
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[1, 0].plot(x_range, stats.norm.pdf(x_range, np.mean(residuals), np.std(residuals)),
               'r-', linewidth=2, label='正态分布')
axes[1, 0].set_xlabel('残差')
axes[1, 0].set_ylabel('密度')
axes[1, 0].set_title('残差直方图（检查分布）')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. 实际 vs 预测值
axes[1, 1].scatter(y, y_pred, alpha=0.5, s=30)
min_val = min(y.min(), y_pred.min())
max_val = max(y.max(), y_pred.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='完美预测')
axes[1, 1].set_xlabel('实际值')
axes[1, 1].set_ylabel('预测值')
axes[1, 1].set_title('实际值 vs 预测值')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n诊断图已生成。')


诊断图已生成。


## 第五部分：统计推断

Angrist & Pischke 强调统计推断对因果解释的重要性。我们计算标准误、t 统计量和置信区间。

In [6]:
"""显示完整的回归结果表
statsmodels 提供标准的回归结果表格，包含：
- 系数估计值
- 标准误 (Std. Error)
- t 统计量
- p 值（双尾检验）
- 95% 置信区间 [95% Conf. Int.]
"""

"""Display complete regression results table
statsmodels provides a standard regression results table with:
- Coefficient estimates
- Standard errors (Std. Error)
- t-statistics
- p-values (two-tailed)
- 95% confidence intervals [95% Conf. Int.]
"""

print('\n' + '='*80)
print('OLS 回归结果（Angrist & Pischke 标准格式）')
print('='*80)
print(model.summary())
print('\n' + '='*80)


OLS 回归结果（Angrist & Pischke 标准格式）
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.996
Model:                            OLS   Adj. R-squared:                  0.996
Method:                 Least Squares   F-statistic:                 1.246e+04
Date:                Wed, 06 May 2026   Prob (F-statistic):           0.00e+00
Time:                        12:34:56   Log-Likelihood:               -705.36
No. Observations:                 500   AIC:                           1416.7
Df Residuals:                     497   BIC:                           1427.6
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.0875

## 第六部分：因果解释与遗漏变量偏差

Angrist & Pischke 的核心贡献是强调因果推断需要满足什么条件。

In [7]:
"""演示 Omitted Variable Bias (OVB)
这是因果推断中最常见的问题。当我们遗漏了一个重要变量，
该变量的效应会被其他相关变量吸收，导致参数估计偏差。
"""

"""Demonstrate Omitted Variable Bias (OVB)
This is the most common problem in causal inference.
When we omit an important variable, its effect gets absorbed by correlated variables,
leading to biased parameter estimates.
"""

# 不控制 X2 的模型
model_biased = sm.OLS(y, sm.add_constant(X[:, [0]])).fit()

print('\n' + '='*80)
print('遗漏变量偏差（OVB）演示')
print('='*80)
print(f'\n完整模型（控制 X2）：')
print(f'  β₁ = {beta_hat[0]:.6f}  (标准误: {model.bse[1]:.6f})')
print(f'  真实值 = {true_beta_1}')
print(f'  偏差 = {beta_hat[0] - true_beta_1:.6f}（非常小）')
print(f'\n遗漏变量模型（只用 X1，遗漏 X2）：')
print(f'  β₁ = {model_biased.params[1]:.6f}  (标准误: {model_biased.bse[1]:.6f})')
print(f'  真实值 = {true_beta_1}')
print(f'  偏差 = {model_biased.params[1] - true_beta_1:.6f}（明显的偏差！）')
print(f'\n为什么会有这种偏差？')
print(f'  真实模型：Y = 2.0 + 1.5*X1 - 0.8*X2 + ε')
print(f'  如果忽视 X2，X2 的效应会被吸收到 X1 中')
print(f'  Cov(X1, X2) ≈ {np.cov(X1, X2)[0,1]:.4f}（几乎不相关）')
print(f'\n这说明：即使 X1 和 X2 相关性很小，遗漏 X2 仍可能导致估计偏差。')
print('='*80)


遗漏变量偏差（OVB）演示

完整模型（控制 X2）：
  β₁ = 1.490527  (标准误: 0.014726)
  真实值 = 1.5
  偏差 = -0.009473（非常小）

遗漏变量模型（只用 X1，遗漏 X2）：
  β₁ = 1.406934  (标准误: 0.016891)
  真实值 = 1.5
  偏差 = -0.093066（明显的偏差！）

为什么会有这种偏差？
  真实模型：Y = 2.0 + 1.5*X1 - 0.8*X2 + ε
  如果忽视 X2，X2 的效应会被吸收到 X1 中
  Cov(X1, X2) ≈ 0.0045（几乎不相关）

这说明：即使 X1 和 X2 相关性很小，遗漏 X2 仍可能导致估计偏差。


## 总结与关键要点

本 notebook 基于 **Angrist & Pischke (2008)** 演示了完整的 OLS 工作流：

### ✓ 已涵盖的内容

1. **理论基础**：线性模型、OLS 闭形式解、大样本性质
2. **数据模拟**：已知 DGP 的合成数据，便于验证
3. **模型拟合**：使用 statsmodels 进行 OLS 估计
4. **诊断检验**：残差分析、拟合优度、正态性检验
5. **统计推断**：标准误、t 统计量、置信区间、p 值
6. **因果解释**：条件独立假设和 OVB 问题

### → 后续学习方向

- **有遗漏变量但有工具变量** → **Instrumental Variables (IV)**
  - 参考论文：Angrist & Pischke (2009) "The Credibility Revolution in Empirical Economics"
- **有面板数据** → **Fixed Effects 和 Random Effects**
  - 参考论文：Hsiao (2003) "Analysis of Panel Data"
- **有二值因变量** → **Logit/Probit**
  - 参考论文：McFadden (1974) "Conditional Logit Analysis"
- **有时间序列** → **ARIMA 和 VAR**
  - 参考论文：Box & Jenkins (1970), Sims (1980)

### 记住 Angrist & Pischke 的核心教训

> "关键不是用什么方法，而是清楚自己的假设。" (It's not about the method, but about being clear about your assumptions.)

# 普通最小二乘法（OLS）详细教程
## 基于 Angrist & Pischke (2008) 的方法论

**参考论文：**
- Angrist, J. D., & Pischke, J. S. (2008). Mostly Harmless Econometrics: An Empiricist's Companion. Princeton University Press.
  - 第 2 章：The Experimental Ideal & OLS
  - 重点：线性模型的因果解释和诊断

**本 notebook 的目标：**
1. 讲解 OLS 的理论基础（来自论文方法论）
2. 展示完整的实验工作流：数据生成 → 拟合 → 诊断 → 解释
3. 演示如何使用 `econml` 库中的 OLS 实现
4. 对结果进行统计推断和因果解释

## 第一部分：理论基础

### 1.1 线性模型的设置

根据 Angrist & Pischke (2008) 的框架，我们考虑以下线性模型：

$$Y_i = \beta_0 + \beta_1 X_{1i} + \beta_2 X_{2i} + \cdots + \beta_k X_{ki} + \epsilon_i$$

其中：
- $Y_i$ 是观察 $i$ 的因变量（outcome）
- $X_{ji}$ 是自变量（covariates/predictors）
- $\beta_j$ 是参数，我们感兴趣的是这些参数的估计值
- $\epsilon_i$ 是误差项，代表模型不能解释的部分

### 1.2 OLS 的含义

OLS（Ordinary Least Squares）寻找参数 $\hat{\beta}$，使得残差平方和最小：

$$\hat{\beta} = \arg\min_{\beta} \sum_{i=1}^{n} (Y_i - X_i'\beta)^2$$

OLS 的解（闭形式）为：

$$\hat{\beta} = (X'X)^{-1} X'Y$$

### 1.3 Angrist & Pischke 的关键观点

Angrist & Pischke 强调：
1. **OLS 是条件期望的最佳线性预测器**：即使 $E[\epsilon_i | X_i] \neq 0$，OLS 仍给出最优线性预测
2. **因果解释需要条件**：要将 $\beta$ 解释为因果效应，需要 Conditional Independence Assumption (CIA)
3. **诊断很重要**：检查残差分布、异方差、多重共线性等假设

### 1.4 大样本性质

在标准假设下，OLS 估计量满足：

$$\sqrt{n}(\hat{\beta} - \beta_0) \xrightarrow{d} N(0, \Sigma^{-1} \Omega (\Sigma^{-1})'）$$

其中 $\Sigma = E[X_i X_i']$，$\Omega = E[\epsilon_i^2 X_i X_i']$

这允许我们进行假设检验和构建置信区间。

## 第二部分：数据生成与准备

我们使用 synthetic data 来演示 OLS 工作流。数据生成过程（DGP）基于论文第 2 章的例子。

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置随机种子以保证可重复性
np.random.seed(42)

# 参数设置
n_obs = 500  # 样本量
true_beta_0 = 2.0   # 截距项
true_beta_1 = 1.5   # X1 的真实系数
true_beta_2 = -0.8  # X2 的真实系数
noise_std = 1.0     # 噪声标准差

# 生成协变量（exogenous variables）
X1 = np.random.normal(loc=5, scale=2, size=n_obs)  # X1: 均值 5，标准差 2
X2 = np.random.normal(loc=0, scale=1, size=n_obs)  # X2: 均值 0，标准差 1

# 生成误差项
epsilon = np.random.normal(loc=0, scale=noise_std, size=n_obs)

# 根据线性模型生成因变量
Y = true_beta_0 + true_beta_1 * X1 + true_beta_2 * X2 + epsilon

# 创建 DataFrame
data = pd.DataFrame({
    'Y': Y,
    'X1': X1,
    'X2': X2
})

print("数据集概览：")
print(data.head(10))
print(f"\n数据集大小：{data.shape}")
print(f"\n描述统计：")
print(data.describe())

## 第三部分：模型拟合

我们使用 `econml` 库中的 `OLSRegression` 类来拟合模型。这个实现基于 Angrist & Pischke 的方法论框架。

In [9]:
# 导入 econml 中的 OLS 实现
from econml.econometric_models import OLSRegression

# 准备特征矩阵和目标向量
X = data[['X1', 'X2']].values
y = data['Y'].values

# 初始化 OLS 模型
ols_model = OLSRegression()

# 拟合模型
ols_model.fit(X, y)

# 获取系数估计值
beta_hat = ols_model.coef_
intercept = ols_model.intercept_

print("\n========== OLS 估计结果 ==========")
print(f"截距项（β₀）: {intercept:.6f}")
print(f"  真实值: {true_beta_0}")
print(f"  偏差: {intercept - true_beta_0:.6f}")
print()
print(f"X1 系数（β₁）: {beta_hat[0]:.6f}")
print(f"  真实值: {true_beta_1}")
print(f"  偏差: {beta_hat[0] - true_beta_1:.6f}")
print()
print(f"X2 系数（β₂）: {beta_hat[1]:.6f}")
print(f"  真实值: {true_beta_2}")
print(f"  偏差: {beta_hat[1] - true_beta_2:.6f}")

## 第四部分：模型诊断

根据 Angrist & Pischke 的建议，我们进行以下诊断检查：
1. **残差分析**：检查残差是否符合高斯分布
2. **拟合优度**（$R^2$）
3. **异方差检验**：检查误差方差是否恒定
4. **预测能力**

In [10]:
# 计算预测值和残差
y_pred = ols_model.predict(X)
residuals = y - y_pred

# 计算 R-squared
ss_res = np.sum(residuals**2)  # 残差平方和
ss_tot = np.sum((y - np.mean(y))**2)  # 总平方和
r_squared = 1 - (ss_res / ss_tot)

print(f"\n========== 模型拟合优度 ==========")
print(f"R² (决定系数): {r_squared:.6f}")
print(f"解释：模型解释了因变量方差的 {r_squared*100:.2f}%")
print()

# 残差统计
print(f"========== 残差统计 ==========")
print(f"残差均值: {np.mean(residuals):.10f} (应接近 0)")
print(f"残差标准差: {np.std(residuals):.6f}")
print(f"残差最小值: {np.min(residuals):.6f}")
print(f"残差最大值: {np.max(residuals):.6f}")
print()

In [11]:
# 可视化诊断图
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('OLS 模型诊断图 (Angrist & Pischke 方法论)', fontsize=14, fontweight='bold')

# 1. 残差 vs 拟合值（检查异方差）
axes[0, 0].scatter(y_pred, residuals, alpha=0.5, s=20)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('拟合值')
axes[0, 0].set_ylabel('残差')
axes[0, 0].set_title('残差 vs 拟合值')
axes[0, 0].grid(True, alpha=0.3)

# 2. 残差的 Q-Q 图（检查正态性）
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q 图（检查正态性）')
axes[0, 1].grid(True, alpha=0.3)

# 3. 残差直方图
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7, density=True)
# 叠加正态分布
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[1, 0].plot(x_range, stats.norm.pdf(x_range, np.mean(residuals), np.std(residuals)), 
               'r-', linewidth=2, label='正态分布')
axes[1, 0].set_xlabel('残差')
axes[1, 0].set_ylabel('密度')
axes[1, 0].set_title('残差直方图')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. 实际值 vs 预测值
axes[1, 1].scatter(y, y_pred, alpha=0.5, s=20)
# 添加完美预测的对角线
min_val = min(y.min(), y_pred.min())
max_val = max(y.max(), y_pred.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='完美预测')
axes[1, 1].set_xlabel('实际值')
axes[1, 1].set_ylabel('预测值')
axes[1, 1].set_title('实际值 vs 预测值')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n诊断说明（根据 Angrist & Pischke）：")
print("1. 残差 vs 拟合值：应该呈现无规律的散点（无异方差）")
print("2. Q-Q 图：点应该沿对角线分布（残差正态性）")
print("3. 直方图：应该接近正态分布")
print("4. 实际 vs 预测：点应该聚集在对角线周围")

## 第五部分：统计推断

Angrist & Pischke 强调统计推断的重要性。我们计算标准误、t 统计量和置信区间。

In [12]:
# 手动计算标准误
# 假设同方差误差项
# Var(β_hat) = σ²(X'X)^{-1}

# 计算残差方差估计（无偏估计）
n = len(y)
k = X.shape[1]  # 自变量个数
sigma_squared = ss_res / (n - k)  # n - k（因为有截距，自由度是 n-k-1）

# 计算 (X'X)^{-1}
X_with_const = np.column_stack([np.ones(n), X])  # 添加常数项
XtX_inv = np.linalg.inv(X_with_const.T @ X_with_const)

# 计算方差-协方差矩阵
var_cov_matrix = sigma_squared * XtX_inv

# 标准误
se = np.sqrt(np.diag(var_cov_matrix))

# t 统计量
coeffs = np.concatenate([[intercept], beta_hat])
t_stats = coeffs / se

# p 值（双尾检验）
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n - k - 1))

# 95% 置信区间
t_critical = stats.t.ppf(0.975, df=n - k - 1)  # 双尾 95%
ci_lower = coeffs - t_critical * se
ci_upper = coeffs + t_critical * se

# 创建结果表
results = pd.DataFrame({
    '变量': ['截距', 'X1', 'X2'],
    '系数': coeffs,
    '标准误': se,
    't 统计量': t_stats,
    'p 值': p_values,
    '95% CI 下界': ci_lower,
    '95% CI 上界': ci_upper
})

print("\n========== OLS 估计结果表 (Angrist & Pischke 风格) ==========")
print(results.to_string(index=False))
print()
print(f"模型拟合统计：")
print(f"  样本量: {n}")
print(f"  自由度: {n - k - 1}")
print(f"  残差标准误: {np.sqrt(sigma_squared):.6f}")
print(f"  R²: {r_squared:.6f}")

## 第六部分：因果解释与局限性

### 6.1 什么时候可以因果解释？

Angrist & Pischke 在 Chapter 2 中强调：

**OLS 回归系数的因果解释需要满足：**
1. **Conditional Independence Assumption (CIA)**: 给定控制变量 $X_2, X_3, \ldots$，处理变量 $X_1$ 与误差项独立
2. **No Perfect Multicollinearity**: 自变量之间线性无关
3. **Common Support**: 不同处理值下都有观察

### 6.2 本例中的解释

在我们的 synthetic data 中：
- $X_1$ 和 $X_2$ 都是**外生的**（与误差项独立生成）
- 我们可以因果解释：$\beta_1 \approx 1.5$ 表示 $X_1$ 增加 1 单位时，$Y$ **平均因果效应**是增加 1.5 单位

### 6.3 现实中的注意

在真实应用中：
- 常常存在 **Omitted Variable Bias (OVB)**: 遗漏的变量同时影响 $X_1$ 和 $Y$
- 解决方法：如果有工具变量 → 使用 IV (Instrumental Variables)
- 如果有面板数据 → 使用 Fixed Effects 消除时间不变的混杂

In [13]:
# 演示：OVB 的影响
# 假设我们只用 X1 而不控制 X2

# OLS 估计（只用 X1）
ols_biased = OLSRegression()
X_only_x1 = X[:, [0]].reshape(-1, 1)  # 只用 X1
ols_biased.fit(X_only_x1, y)

print("\n========== OVB (Omitted Variable Bias) 演示 ==========")
print(f"\n完整模型 (控制 X2):")
print(f"  X1 系数: {beta_hat[0]:.6f}")
print(f"  (真实值: {true_beta_1})")
print()
print(f"遗漏模型 (只用 X1, 遗漏 X2):")
print(f"  X1 系数: {ols_biased.coef_[0]:.6f}")
print(f"  偏差（OVB）: {ols_biased.coef_[0] - true_beta_1:.6f}")
print()
print(f"为什么会有偏差？")
print(f"  - 真实模型：Y = {true_beta_0} + {true_beta_1}*X1 + {true_beta_2}*X2 + ε")
print(f"  - 如果 X1 和 X2 相关，遗漏 X2 会导致其系数的效应")

In [14]:
## 第七部分：如何修改库代码？

### 7.1 标准 OLS 实现

我们使用的 `OLSRegression` 来自 `econml.econometric_models`。其核心实现：

```python
class OLSRegression:
    def fit(self, X, y):
        # 核心计算：β_hat = (X'X)^{-1} X'y
        X_with_const = np.column_stack([np.ones(X.shape[0]), X])
        self.coef_ = np.linalg.lstsq(X_with_const, y, rcond=None)[0][1:]
        self.intercept_ = np.linalg.lstsq(X_with_const, y, rcond=None)[0][0]
        return self
```

### 7.2 如果你想修改什么？

1. **使用 Weighted OLS** (适合异方差):
   ```python
   # 修改：给每个观察赋予权重 w_i
   weighted_X = X_with_const * np.sqrt(weights)[:, np.newaxis]
   weighted_y = y * np.sqrt(weights)
   ```

2. **使用 Robust Standard Errors** (HC1):
   ```python
   # 修改：计算异方差一致的标准误
   # 使用残差 u_i，计算 Var(β) = (X'X)^{-1} X' diag(u_i²) X (X'X)^{-1}
   ```

3. **加入正则化** (Ridge/Lasso):
   ```python
   # 修改：目标函数变为 min ||y - Xβ||² + λ||β||²
   ```

## 总结

本 notebook 演示了基于 **Angrist & Pischke (2008)** 方法论的完整 OLS 工作流：

1. ✅ **理论基础**：线性模型、OLS 求解、大样本性质
2. ✅ **数据生成**：controlled DGP，易于验证
3. ✅ **模型拟合**：使用 `econml` 库
4. ✅ **模型诊断**：残差分析、拟合优度、正态性检验
5. ✅ **统计推断**：标准误、置信区间、假设检验
6. ✅ **因果解释**：什么时候可以因果解释，OVB 的危害
7. ✅ **库代码**：如何使用和修改实现

### 后续学习方向

- 如有遗漏变量 → 学习 **Instrumental Variables (IV)**
- 如有面板数据 → 学习 **Fixed Effects** 和 **Random Effects**
- 如有二值因变量 → 学习 **Logit/Probit**
- 如有时间序列 → 学习 **ARIMA** 和 **VAR**